# TirraMind — HetTGN GNN Retrain on Kaggle or CPU

This notebook resumes training from the **latest epoch checkpoint** found in the dataset and runs up to `--epochs 40` total epochs.

**Current state:** 20 epochs complete. Next run targets epochs 21–40 (Phase 42: enriched entity graph — 19 cftc_tracks + 119 produced_in links).

## Before Running — Two Things to Do

### Step 1 — Make sure two datasets are attached

**Dataset 1: `tirramind-data`**
It must contain a directory with:
- `pipeline.db`  (enriched: 19 cftc_tracks + 119 produced_in links)
- `checkpoints/epoch_020.pt`  (latest checkpoint)

**Dataset 2: `tirramind-code`** (fallback only — not needed if `tirramind_token` secret is set)

The notebook does **not** read zip files at runtime. It searches the mounted Kaggle dataset tree for those extracted directories/files directly.

### Step 2 — Attach datasets to THIS notebook

In the Kaggle notebook editor, right panel → **Data** → **Add Dataset**:
- Search `tirramind-data` → Add

Optional but recommended: set **Accelerator → GPU T4 x1** in Settings.

### Step 3 — GitHub Token Secret (for always-latest code)

Add a Kaggle Secret named `tirramind_token` with a fine-grained PAT with **read** access to `savabs/tirramind`.
Kaggle → Settings → Secrets → Add New Secret

---
After training, download `gnn_model.pt` and the new epoch checkpoints from the **Output** tab.


## 1. Install and Import Required Libraries

Install `torch-geometric` (and its sparse/scatter kernels) matching the Kaggle-provided PyTorch version.



In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

# torch-geometric — core dependency not bundled in Kaggle's default image
pip("torch-geometric==2.7.0")

# scatter/sparse kernels — pick wheels matching the active torch + device runtime
import torch

torch_ver = torch.__version__.split("+")[0]
cuda_runtime = torch.version.cuda
cuda_tag = f"cu{cuda_runtime.replace('.', '')}" if cuda_runtime else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print(f"Installing PyG extras for torch={torch_ver}, runtime={cuda_tag}")
pip("torch-scatter", "torch-sparse", "-f", wheel_url)

# other lightweight deps
pip("tqdm", "rich")

# W&B — real-time loss streaming dashboard
pip("wandb")

print("All dependencies installed.")


## 2. Setup Working Directory

**Code** is cloned directly from GitHub (always latest — no dataset re-upload needed for code changes).
To enable this, add a Kaggle Secret named `tirramind_token` with a fine-grained personal access token
that has **read** access to the `tirramind_v1` repo.

Go to: Kaggle → Settings → Secrets → Add New Secret → Name: `tirramind_token`

If no secret is set, falls back to the `tirramind-code` dataset (backward compatible).

**Data** (`pipeline.db` + epoch checkpoints) is still read from the `tirramind-data` dataset —
these are large binary files that change infrequently and are not in git.



In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. CODE: git clone (preferred) or fall back to dataset ───────────────────
GITHUB_REPO = "savabs/tirramind"   # GitHub username/repo

_cloned_from_git = False
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("tirramind_token")
    _repo_url = f"https://{_token}@github.com/{GITHUB_REPO}.git"

    # Fresh clone every run — always gets the latest trainer.py / scripts
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(
        ["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
        check=True,
        capture_output=True,
    )
    print(f"✓ Cloned {GITHUB_REPO} → {WORK_DIR}  (latest commit, no dataset re-upload needed)")
    _cloned_from_git = True

except Exception as _e:
    print(f"Git clone skipped ({_e.__class__.__name__}: {_e})")
    print("  Falling back to tirramind-code dataset — add tirramind_token secret to avoid this.")

    def find_code_root(root: str = "/kaggle/input") -> Path | None:
        for dirpath, dirs, _ in os.walk(root):
            if {"agent", "scripts"}.issubset(set(dirs)):
                return Path(dirpath)
        return None

    code_root = find_code_root()
    assert code_root is not None, (
        "No tirramind_token secret and no tirramind-code dataset found. "
        "Either add the secret or attach the dataset."
    )
    for name in ("agent", "scripts"):
        dest = WORK_DIR / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(code_root / name, dest)
        print(f"  Copied {name}/ from dataset")

# ── 2. DATA: pipeline.db + checkpoints always come from the dataset ──────────
def find_data_root(root: str = "/kaggle/input") -> Path | None:
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files) and "checkpoints" in set(dirs):
            return Path(dirpath)
    return None

data_root = find_data_root()
assert data_root is not None, (
    "Could not find tirramind-data dataset containing pipeline.db and checkpoints/. "
    "Attach the dataset in the right panel → Data → Add Dataset."
)

pipeline_db = data_root / "pipeline.db"
ckpt_src    = data_root / "checkpoints"
checkpoint_files = sorted(ckpt_src.glob("epoch_*.pt"))
assert checkpoint_files, f"No epoch_*.pt files found in {ckpt_src}"

pipeline_dir = WORK_DIR / ".tirra_pipeline"
ckpt_dir     = pipeline_dir / "checkpoints"
pipeline_dir.mkdir(exist_ok=True)
ckpt_dir.mkdir(exist_ok=True)

shutil.copy2(pipeline_db, pipeline_dir / "pipeline.db")
print(f"✓ pipeline.db  ({pipeline_db.stat().st_size // 1_000_000} MB)")

for ckpt in checkpoint_files:
    shutil.copy2(ckpt, ckpt_dir / ckpt.name)
    print(f"✓ {ckpt.name}  ({ckpt.stat().st_size // 1_000_000} MB)")

# ── 3. Patch pipeline __init__ to avoid eager APScheduler import ─────────────
pipeline_init = WORK_DIR / "agent" / "pipeline" / "__init__.py"
pipeline_init.write_text(
    '"""TirraMind — Pipeline Layer (Deterministic DAG Scheduler)."""\n\n'
    "from agent.pipeline.storage_backend import (\n"
    "    PostgresBackend,\n    SQLiteBackend,\n    StorageBackend,\n)\n"
    "from agent.pipeline.store import PipelineStore\n\n"
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n\n'
    "def __getattr__(name: str):\n"
    '    if name == "PipelineScheduler":\n'
    "        from agent.pipeline.scheduler import PipelineScheduler\n\n"
    "        return PipelineScheduler\n"
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8",
)
print("✓ Patched agent.pipeline.__init__.py for lazy scheduler import")
print("\nSetup complete.")


## 3. Environment Check — GPU, PyTorch, PyG Versions


In [ ]:
import torch
import torch_geometric

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU VRAM     : {total_mem} GB")
else:
    print("GPU name     : none")
    print("GPU VRAM     : 0 GB")
print(f"PyG          : {torch_geometric.__version__}")
print(f"Device       : {DEVICE}")

if DEVICE == "cpu":
    print("Running in CPU fallback mode. This is supported, but it will be slower.")


## 4. Pre-flight Check — Verify Required Files Exist


In [ ]:
from pathlib import Path
import shutil

WORK_DIR    = Path("/kaggle/working/tirramind_v1")
PIPELINE_DB = WORK_DIR / ".tirra_pipeline" / "pipeline.db"

# H-H: hypothesis-specific checkpoint subfolder
SHARED_CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints"
CKPT_DIR        = SHARED_CKPT_DIR / "h_h"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Seed: find latest epoch checkpoint from the shared (zip) folder
shared_checkpoints = sorted(SHARED_CKPT_DIR.glob("epoch_*.pt"))
assert shared_checkpoints, (
    f"No seed checkpoints in {SHARED_CKPT_DIR}. "
    "Upload the zip containing epoch_0*.pt before running."
)
latest_shared = shared_checkpoints[-1]
seed_epoch    = int(latest_shared.stem.split("_")[1])  # epoch_030 -> 30

# Copy seed into h_h/ if not already there
seed_dest = CKPT_DIR / latest_shared.name
if not seed_dest.exists():
    shutil.copy2(latest_shared, seed_dest)
    print(f"Seeded {latest_shared.name} → checkpoints/h_h/")

resume_epoch = seed_epoch
print(f"Latest checkpoint : {latest_shared.name}  ({latest_shared.stat().st_size // 1_000_000} MB)")
print(f"H-H checkpoint dir: {CKPT_DIR}")

# Verify pipeline DB
assert PIPELINE_DB.exists(), f"pipeline.db not found at {PIPELINE_DB}"
print(f"pipeline.db       : {PIPELINE_DB.stat().st_size // 1_000_000} MB")

# Verify key source files used by retrain_gnn.py and Trainer
for rel_path in [
    "agent/models/gnn/graph_builder.py",
    "agent/models/gnn/het_tgn.py",
    "agent/models/gnn/trainer.py",
    "agent/models/gnn/temporal.py",
    "agent/models/gnn/ewc.py",
    "agent/models/gnn/alignment.py",
    "scripts/retrain_gnn.py",
    "agent/pipeline/store.py",
]:
    p = WORK_DIR / rel_path
    assert p.exists(), f"Missing source file: {rel_path}"
    print(f"  ✓ {rel_path}")

print(f"\nAll checks passed. H-H will resume from epoch {resume_epoch}.")
RESUME_EPOCH = resume_epoch


## 5. Run GNN Training — H-H: Vessel → Commodity Links

Runs `scripts/add_vessel_commodity_links.py` to add `carries_commodity`
links from vessel entities to instrument entities, then trains with same
architecture as H-A.

**Hypothesis H-H**: AIS vessel type (tanker/cargo) creates L3 cross-entity
edges connecting physical ship movements directly to commodity instruments
(CL=F, BZ=F, ZC=F, ZW=F). The GNN can propagate positioning signal across
these new edges → higher commodity IC.

> Architecture: `--num-layers 2 --num-heads 2 --hidden-dim 128` (same as H-A)
> Resumes from epoch 30. Vessel→commodity links added before training.


In [ ]:
import subprocess
import sys
import os
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/tirramind_v1")
CKPT_DIR     = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "h_h"
PIPELINE_DB  = WORK_DIR / ".tirra_pipeline" / "pipeline.db"
DEVICE       = globals().get("DEVICE", "cpu")

resume_epoch = globals().get("RESUME_EPOCH")
if resume_epoch is None:
    checkpoints = sorted(CKPT_DIR.glob("epoch_*.pt"))
    assert checkpoints, f"No checkpoints found in {CKPT_DIR}"
    resume_epoch = int(checkpoints[-1].stem.split("_")[1])
    print(f"Recovered RESUME_EPOCH={resume_epoch} from checkpoint directory.")

TARGET_EPOCHS = 40   # H-H: epochs 31-40 with vessel→commodity links

# ── W&B: read API key from Kaggle secret ─────────────────────────────────────
_wandb_project = None
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = _wandb_key
    _wandb_project = "tirramind"
    print(f"W&B enabled → project '{_wandb_project}' (run: h-h-ep{resume_epoch+1}-{TARGET_EPOCHS})")
except Exception as _e:
    print(f"W&B disabled (no WANDB_API_KEY secret: {_e.__class__.__name__})")

print(f"Using device: {DEVICE}")
print(f"H-H: resuming from epoch {resume_epoch}, training to epoch {TARGET_EPOCHS}")

# ── Step 1: Add vessel → commodity links ──────────────────────────────────────
print("\n[H-H] Adding vessel → commodity links to DB ...")
link_cmd = [
    sys.executable, "scripts/add_vessel_commodity_links.py",
    "--db-path", str(PIPELINE_DB),
]
link_result = subprocess.run(
    link_cmd, cwd=str(WORK_DIR), capture_output=True, text=True
)
print(link_result.stdout)
if link_result.returncode != 0:
    print("WARNING: vessel link addition failed (non-fatal):", link_result.stderr)
else:
    print("[H-H] Vessel → commodity links written successfully.")

# ── Step 2: Train ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable, "scripts/retrain_gnn.py",
    "--epochs",              str(TARGET_EPOCHS),
    "--hidden-dim",          "128",
    "--num-layers",          "2",
    "--num-heads",           "2",
    "--lr",                  "1e-3",
    "--backup",
    "--window-size",         "604800",
    "--gdelt-frac",          "0.05",
    "--max-windows",         "200",
    "--auto-tune",
    "--listnet",
    "--return-log-var-max",  "0.0",
    "--device",              DEVICE,
    "--skip-eval",
    "--checkpoint-dir",      str(CKPT_DIR),
    "--model-out",           ".tirra_pipeline/gnn_model_h_h.pt",
    "--resume",              str(resume_epoch),
]

if _wandb_project:
    cmd += [
        "--wandb-project", _wandb_project,
        "--wandb-run",     f"h-h-ep{resume_epoch+1}-{TARGET_EPOCHS}",
        "--wandb-tags",    "h-h,vessel-links,phase-43",
    ]

print("Running:", " ".join(cmd))
print("-" * 70)

process = subprocess.Popen(
    cmd,
    cwd=str(WORK_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print("H-H training completed successfully.")


## 6. Verify Outputs and Prepare for Download


In [ ]:
import shutil
from pathlib import Path

WORK_DIR   = Path("/kaggle/working/tirramind_v1")
CKPT_DIR   = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "h_h"
OUT_DIR    = Path("/kaggle/working")

# Epochs already on the laptop before this run — skip re-copying those
ALREADY_HAVE_UP_TO = resume_epoch  # set by Cell 9 above

print("=== H-H checkpoints saved ===")
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    size_mb = ckpt.stat().st_size / 1_000_000
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

# Copy final model
final_model = WORK_DIR / ".tirra_pipeline" / "gnn_model_h_h.pt"
if final_model.exists():
    shutil.copy2(final_model, OUT_DIR / "gnn_model_h_h.pt")
    print(f"\ngnn_model_h_h.pt → /kaggle/working/ ({final_model.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("\ngnn_model_h_h.pt not found — training may not have completed yet.")

# Copy only NEW checkpoints (those produced this run)
new_checkpoints = []
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    epoch_num = int(ckpt.stem.split("_")[1])
    if epoch_num > ALREADY_HAVE_UP_TO:
        dest = OUT_DIR / ckpt.name
        shutil.copy2(ckpt, dest)
        new_checkpoints.append(ckpt.name)
        print(f"{ckpt.name} → /kaggle/working/")

print("\n=== Download from the 'Output' tab ===")
print("Files to grab:")
print("  gnn_model_h_h.pt")
for name in new_checkpoints:
    print(f"  {name}")


## 7. Copy H-H Results Back to Laptop

After the Kaggle session finishes, download the output files and run:

```bash
cd /home/becmachlean/2024/projects/tirramind_v1

# Save H-H final model
cp ~/Downloads/gnn_model_h_h.pt .tirra_pipeline/gnn_model_h_h.pt

# Copy new epoch checkpoints into the h_h/ subfolder
mkdir -p .tirra_pipeline/checkpoints/h_h/
for f in ~/Downloads/epoch_*.pt; do
    [ -f "$f" ] && cp "$f" .tirra_pipeline/checkpoints/h_h/
done

# Verify
ls -lh .tirra_pipeline/checkpoints/h_h/
ls -lh .tirra_pipeline/gnn_model_h_h.pt

# Run IC diagnostic backtest
python scripts/phase40_gnn_backtest.py --model .tirra_pipeline/gnn_model_h_h.pt

# Compare H-H vs H-A baseline (did vessel→commodity links improve IC?)
python scripts/compare_experiments.py --latest 2
```

**Next run:** update `TARGET_EPOCHS` in Cell 11 to the new target (e.g. 50).
The notebook auto-detects the latest checkpoint in `checkpoints/h_h/` and resumes from there.


## 8. Backtest H_H Model — IC / Sharpe vs Equal-Weight Baseline


In [ ]:
import subprocess, sys, shutil
from pathlib import Path

WORK_DIR   = Path("/kaggle/working/tirramind_v1")
PIPELINE   = WORK_DIR / ".tirra_pipeline"
MODEL_HYP  = PIPELINE / "gnn_model_h_h.pt"
MODEL_LINK = PIPELINE / "gnn_model.pt"   # backtest expects this name
OUT_DIR    = Path("/kaggle/working")

assert MODEL_HYP.exists(), f"Model not found: {MODEL_HYP}"

# backtest.py uses hardcoded path .tirra_pipeline/gnn_model.pt relative to cwd
if MODEL_LINK.exists():
    MODEL_LINK.unlink()
shutil.copy2(MODEL_HYP, MODEL_LINK)
print(f"Linked gnn_model_h_h.pt -> gnn_model.pt")

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts/phase40_gnn_backtest.py")],
    cwd=str(WORK_DIR),
    text=True,
)

# Copy experiment manifest(s) to /kaggle/working/ for download
exp_dir = PIPELINE / "experiments"
if exp_dir.exists():
    for f in sorted(exp_dir.glob("exp_*.json"))[-3:]:
        shutil.copy2(f, OUT_DIR / f.name)
        print(f"Saved {f.name} -> /kaggle/working/")

if result.returncode != 0:
    print("
Backtest FAILED")
else:
    print("
Backtest complete — download exp_*.json from Output tab.")
